# 01 · Motivation (a) — Phase heterogeneity

**Toy example 1.** Five vehicles respond to the same stimulus with the same
amplitude but *different phase shifts* θᵢ:

$$ r_i(x) = \sin\big(2\pi (x + \theta_i)\big). $$

A single *global* model fitted by least squares to all vehicles collapses to a
constant (the responses cancel), i.e. the global model learns nothing about any
individual. The two-shot decomposition recovers each vehicle's behavior as a
common component $C\Psi$ plus a personalized component $P\Phi$.

**Figures produced:** `Aggregate.png`, `Motivation_a_personalized.png`,
`Motivation_a_common.png`, `Motivation_a_unique.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bhd").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.optimize import minimize
from sklearn.decomposition import PCA

from bhd import FIG_DIR, set_style

set_style()

## 1. Global (population-level) model

In [ ]:
np.random.seed(42)
num_vehicles = 5
x = np.linspace(0, 1, 500)  # normalized stimulus / time axis
theta_values = np.random.uniform(0, 1, num_vehicles)  # random phase shifts
amplitudes = np.ones(num_vehicles)  # constant amplitude for all vehicles
responses = [amp * np.sin(2 * np.pi * (x + theta)) for amp, theta in zip(amplitudes, theta_values)]


def global_model(x, w):
    """Global model: a constant."""
    return w[0] * np.ones_like(x)


def loss_function(w):
    """Mean squared error between the global model and every vehicle response."""
    total_loss = 0
    for amp, theta in zip(amplitudes, theta_values):
        true_response = amp * np.sin(2 * np.pi * (x + theta))
        total_loss += np.mean((global_model(x, w) - true_response) ** 2)
    return total_loss


result = minimize(loss_function, [0.0], method="BFGS")
optimal_w = result.x
print(f"Optimized constant value (w[0]): {optimal_w[0]:.4f}")
print(f"Final loss: {loss_function(optimal_w):.6f}")

plt.figure(figsize=(10, 7))
colors = cm.viridis(np.linspace(0, 1, num_vehicles))
for i, response in enumerate(responses):
    plt.plot(x, response, color=colors[i], alpha=0.25, linewidth=7)
plt.plot([], [], color="black", linewidth=7, alpha=0.25, label="Vehicle i")
plt.plot(x, global_model(x, optimal_w), color="black", linewidth=4, linestyle="-", label="Global Model")
plt.xlabel("x", fontsize=20)
plt.xticks(fontsize=18)
plt.ylabel("Response", fontsize=20)
plt.yticks(fontsize=18)
plt.ylim(-1.3, 1.1)
plt.legend(fontsize=20, loc="lower center", ncol=2)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Aggregate.png", dpi=300, bbox_inches="tight")
plt.show()

## 2. Two-shot personalized decomposition

Shot 1: PCA over the stacked responses gives the shared component.
Shot 2: a per-vehicle PCA on the residual gives the personalized component.

In [ ]:
D_matrix = np.column_stack(responses)  # (n_points, n_agents)

# Shot 1 — shared component
pca_shared = PCA(n_components=1)
Psi_shared = pca_shared.fit_transform(D_matrix)
C_shared = pca_shared.components_.T
D_shared = Psi_shared @ C_shared.T

# Shot 2 — personalized residual per vehicle
personalized_predictions = []
for i in range(num_vehicles):
    R_i = D_matrix[:, i] - D_shared[:, i]
    pca_personal = PCA(n_components=1)
    Phi_i = pca_personal.fit_transform(R_i.reshape(-1, 1))
    P_i = pca_personal.components_
    R_hat_i = Phi_i @ P_i
    personalized_predictions.append(D_shared[:, i] + R_hat_i.flatten())

plt.figure(figsize=(10, 7))
for i, response in enumerate(responses):
    plt.plot(x, response, color=colors[i], alpha=0.25, linewidth=7)
for i, pred in enumerate(personalized_predictions):
    plt.plot(x, pred, linestyle=":", linewidth=4, color=colors[i])
plt.plot([], [], color="black", linewidth=7, alpha=0.25, label="Vehicle i")
plt.plot([], [], linestyle=":", color="black", linewidth=4, label="Personalized Prediction i")
plt.xlabel("x", fontsize=22)
plt.xticks(fontsize=20)
plt.ylabel("Response", fontsize=22)
plt.yticks(fontsize=20)
plt.ylim(-1.3, 1.1)
plt.legend(fontsize=22, loc="lower center", ncol=3)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Motivation_a_personalized.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Common vs. personalized components

In [ ]:
plt.figure(figsize=(10, 7))
for i in range(num_vehicles):
    plt.plot(x, D_shared[:, i], color=colors[i], alpha=0.4, linewidth=7)
plt.xlabel("x", fontsize=22)
plt.xticks(fontsize=20)
plt.ylabel("Response", fontsize=22)
plt.yticks([-1.0, -0.5, 0, 0.5, 1.0], fontsize=20)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Motivation_a_common.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 7))
for i in range(num_vehicles):
    personalized_residual = np.array(personalized_predictions[i]) - D_shared[:, i]
    plt.plot(x, personalized_residual, linestyle="--", color=colors[i], alpha=0.4, linewidth=7)
plt.xlabel("x", fontsize=22)
plt.xticks(fontsize=20)
plt.ylabel("Response", fontsize=22)
plt.yticks([-1.0, -0.5, 0, 0.5, 1.0], fontsize=20)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Motivation_a_unique.png", dpi=300, bbox_inches="tight")
plt.show()